In [1]:
import math
import numpy as np
import matplotlib.pyplot as plt
from random import randrange

class Chebyshev:
    """
    Chebyshev(a, b, n, func)
    Given a function func, lower and upper limits of the interval [a,b],
    and maximum degree n, this class computes a Chebyshev approximation
    of the function.
    Method eval(x) yields the approximated function value.
    """
    def __init__(self, a, b, n, func):
        n = n + 1
        self.a = a
        self.b = b
        self.func = func

        bma = 0.5 * (b - a)
        bpa = 0.5 * (b + a)
        f = [func(math.cos(math.pi * (k + 0.5) / n) * bma + bpa) for k in range(n)]
        self.roots = f

        self.x = [math.cos(math.pi * (k + 0.5) / n) * bma + bpa for k in range(n)]

        fac = 2.0 / n
        self.c = [fac * sum([f[k] * math.cos(math.pi * j * (k + 0.5) / n)
                  for k in range(n)]) for j in range(n)]

    def eval(self, x):
        a,b = self.a, self.b
        #assert(a <= x <= b)
        y = (2.0 * x - a - b) * (1.0 / (b - a))
        y2 = 2.0 * y
        (d, dd) = (self.c[-1], 0)             # Special case first step for efficiency
        for cj in self.c[-2:0:-1]:            # Clenshaw's recurrence
            (d, dd) = (y2 * d - dd + cj, d)
        return y * d - dd + 0.5 * self.c[0]   # Last step is different

In [2]:
def dec(bits):
    return sum(int(bit) << i for i, bit in enumerate(np.round(bits, decimals=2)))

def tobin(x, size):
    x_bin = bin(x)[2:].zfill(size)
    x_bin = [int(bit) for bit in reversed(x_bin)]
    
    sigma = 2**(-30)  # CKKS error
    sigma = 0
    
    return np.array(x_bin) + (np.random.randn(len(x_bin)) * sigma)

In [47]:
def koggestone(a, b, size = 2):
    n = size

    P = mod2shallow(np.add(a, b))
    G = np.multiply(a, b)
    
    Pcopy = P.copy()
    
    step = 1
    
    while step < n:   
        G = G + P * np.roll(G, step)
        P = P * np.roll(P, step)
           
        step *= 2

    carry = np.roll(G, 1)
    
    S = (np.subtract(np.subtract(a, b) ** 2, carry) ** 2)
    
    return S

In [48]:
def mod_reduction(bit_position):
    period = 2**bit_position
    
    assert period % 2 == 0 and period >= 2
    P = period
    seq = np.array([0]*(P//2) + [1]*(P//2), dtype=float)

    c = np.fft.fft(seq) / P

    def f(x):
        s = c[0].real
        for k in range(1, P//2):
            ck = c[k]
            ang = 2 * math.pi * k * x / P
            s += 2 * (ck.real * math.cos(ang) - ck.imag * math.sin(ang))

        s += c[P//2].real * math.cos(math.pi * x)
        return s
    
    return f

In [49]:
a=0
b=225

p1 = Chebyshev(a, b, 493, mod_reduction(1))           # Period 2 {0, 1}
p2 = Chebyshev(a, b, 493, mod_reduction(2))           # Period 4 {0, 0, 1, 1}
p3 = Chebyshev(a, b, 493, mod_reduction(3))           # Period 8 {0, 0, 0, 0, 1, 1, 1, 1}
p4 = Chebyshev(a, b, 493, mod_reduction(4))           # Period 16 
p5 = Chebyshev(a, b, 493, mod_reduction(5))           # Period 32
p6 = Chebyshev(a, b, 493, mod_reduction(6))           # Period 64
p7 = Chebyshev(a, b, 493, mod_reduction(7))           # Period 128
p8 = Chebyshev(a, b, 493, mod_reduction(8))           # Period 256

In [50]:
import math

def simd_bintodec(vec, rep):
    mask = np.array([])
    
    for i in range(rep):
        mask = np.append(mask, np.array([1, 2, 4, 8, 0, 0, 0, 0]))
    
    res = np.multiply(vec, mask)
    res = np.add(res, np.roll(res, -1))
    res = np.add(res, np.roll(res, -2))
    
    mask = np.array([])
    
    for i in range(rep):
        mask = np.append(mask, np.array([1, 0, 0, 0, 0, 0, 0, 0]))
        
    res = np.multiply(res, mask)
    res = np.add(res, np.roll(res, 1))
    res = np.add(res, np.roll(res, 2))
    res = np.add(res, np.roll(res, 4))

    return res

In [51]:
"""
Takes as input a repeated (8 times) vector containing the result of the computation
"""

def simd_4bits_multiplier_parallel(x, y, rep):
    res = np.multiply(x, y)

    for i in range(rep):
        res[i * 8 + 0] = p1.eval(res[i * 8 + 0])
        res[i * 8 + 1] = p2.eval(res[i * 8 + 1])
        res[i * 8 + 2] = p3.eval(res[i * 8 + 2])
        res[i * 8 + 3] = p4.eval(res[i * 8 + 3])
        res[i * 8 + 4] = p5.eval(res[i * 8 + 4])
        res[i * 8 + 5] = p6.eval(res[i * 8 + 5])
        res[i * 8 + 6] = p7.eval(res[i * 8 + 6])
        res[i * 8 + 7] = p8.eval(res[i * 8 + 7])
    
    return res

In [52]:
def mod2(x):
    return x**2*(x - 2)**2

def mod2shallow(x):
    return -x**2+2*x #(or reduce2)

def csa4(A, B, C, D, bits):
    S1, C1 = csa3_bits(A, B, C) 
    C1_shift = roll_shift_left(C1)
    S2, C2 = csa3_bits(S1, C1_shift, D)
    C2_shift = roll_shift_left(C2)
    result = koggestone(S2, C2_shift, bits)
    return result

def majority_bit_poly(a, b, c):
    total = np.add(np.add(a, b), c)
    return (-1/3)*total**3 + (3/2)*total**2 - (7/6)*total

def csa3_bits(a, b, c):
    S = mod2shallow(np.add(a, b))
    S = mod2shallow(np.add(S, c))
    
    C = majority_bit_poly(a, b, c)
    return S, C

def roll_shift_left(bits):
    shifted = np.roll(bits, 1)
    return shifted

In [53]:
def simd_nbits_multiplier(a, b, n, rep = 1, provina = 300):
    
    rep_size = n**2 // 2
    
    dunn = n**2 // 8
    
    if n > 8:
        mask_low = np.zeros(len(a), dtype = int)
        for j in range(rep):
            for i in range(n//2):
                mask_low[(rep_size * j) + i] = 1

        mask_high = np.zeros(len(a), dtype = int)
        for j in range(rep):
            for i in range(n//2, n):
                mask_high[(rep_size * j) + i] = 1
        
        a_processed = np.zeros(len(a), dtype = int)        
        a_processed = np.add(a_processed, np.multiply(a, mask_low))
        a_processed = np.add(a_processed, np.roll(np.multiply(a, mask_low), dunn))
        a_processed = np.add(a_processed, np.roll(np.multiply(a, mask_high), dunn*2-n//2))
        a_processed = np.add(a_processed, np.roll(np.multiply(a, mask_high), dunn*3-n//2))
        
        b_processed = np.zeros(len(b), dtype = int)
        b_processed = np.add(b_processed, np.multiply(b, mask_low))
        b_processed = np.add(b_processed, np.roll(np.multiply(b, mask_low), dunn*2))
        b_processed = np.add(b_processed, np.roll(np.multiply(b, mask_high), dunn-n//2))
        b_processed = np.add(b_processed, np.roll(np.multiply(b, mask_high), dunn*3-n//2))

        
    else:
        # In case of n=8 we do not need to mask as it will be done by 4 bits multiplier
        a_processed = np.zeros(len(a), dtype = int)        
        a_processed = np.add(a_processed, a)
        a_processed = np.add(a_processed, np.roll(a, 8))
        a_processed = np.add(a_processed, np.roll(a, 8+4))
        a_processed = np.add(a_processed, np.roll(a, 16+4))

        b_processed = np.zeros(len(b), dtype = int)
        b_processed = np.add(b_processed, b)
        b_processed = np.add(b_processed, np.roll(b, 16))
        b_processed = np.add(b_processed, np.roll(b, 4))
        b_processed = np.add(b_processed, np.roll(b, 20))
    
    if n == 8:
        mult = simd_4bits_multiplier_parallel(simd_bintodec(a_processed, rep * 4), simd_bintodec(b_processed, rep * 4), rep * 4)
    else:
        mult = simd_nbits_multiplier(a_processed, b_processed, n//2, 4 * rep)

    dunn2 = dunn * 2
    
    mask1 = np.zeros(len(a), dtype=float)
    for j in range(rep):
        for i in range(n):
            mask1[(j * rep_size) + i] = 1
            mask1[(j * rep_size) + i + dunn2] = 1
            
    p1 = np.multiply(mult, mask1)
    p2 = np.roll(p1, -rep_size//2 + n//2)
    
    mask2 = np.zeros(len(a), dtype=float)
    for j in range(rep):
        for i in range(n):
            mask2[(j * rep_size) + rep_size // 4 + i] = 1
            mask2[(j * rep_size) + rep_size // 4 + i + dunn2] = 1
    
    if n == 8:
        p3 = np.roll(np.multiply(mult, mask2), -16)
    else:
        p3 = np.roll(np.multiply(mult, mask2), -rep_size // 4 + n//2)


    if n == 8:
        p4 = np.roll(p3, 12)
    if n == 16:
        p4 = np.roll(np.multiply(mult, mask2), -80)
    if n == 32:
        p4 = np.roll(np.multiply(mult, mask2), -352) 
    if n == 64:
        p4 = np.roll(np.multiply(mult, mask2), -1472) 
        
    return csa4(p1, p2, p3, p4, n)

In [54]:
x = randrange(2**8)
y = randrange(2**8)

print(x * y)
print(dec(simd_nbits_multiplier(tobin(x, 8*4), tobin(y, 8*4), 8, 1)[:16]))

19976
19976


In [55]:
x = randrange(2**16)
y = randrange(2**16)

print(x * y)
print(dec(simd_nbits_multiplier(tobin(x, 16*8), tobin(y, 16*8), 16, 1)[:32]))

891839868
891839868


In [56]:
x = randrange(2**32)
y = randrange(2**32)

print(x * y)
print(dec(simd_nbits_multiplier(tobin(x, 32*16), tobin(y, 32*16), 32, 1)[:64]))

1230609340235766906
1230609340235766906


In [57]:
x = randrange(2**64)
y = randrange(2**64)

print(x * y)
print(dec(simd_nbits_multiplier(tobin(x, 64*32), tobin(y, 64*32), 64, 1)[:128]))

267586891311609488656656690264200551467
267586891311609488656656690264200551467
